# Constrained SLM — Pipeline Walkthrough

Run this after completing Steps 1–6 in the README (venv + install + data in `data/raw/`).
This notebook mirrors the scripts so you can inspect each stage interactively for the write-up.

In [ ]:
import sys, os
# make project root importable from the notebooks/ folder
sys.path.insert(0, os.path.abspath('..'))
from src.config import RAW_DIR, TOP_K
print('raw data dir:', RAW_DIR)

## 1. Extract + chunk

In [ ]:
from src.extract import iter_documents
from src.chunk import chunk_document

docs = list(iter_documents(RAW_DIR))
chunks = [c for d in docs for c in chunk_document(d)]
print(f'{len(docs)} documents -> {len(chunks)} chunks')
chunks[0]

## 2. Build the retrieval index (dense + lexical)

In [ ]:
from src.retriever import build_index, Retriever
build_index(chunks)
retriever = Retriever()

## 3. Compare retrieval methods

In [ ]:
q = 'What is the CIA triad in computer security?'
for method in ['dense', 'lexical', 'hybrid']:
    print(f'\n=== {method} ===')
    for r in retriever.search(q, top_k=3, method=method):
        print(f"  {r['score']:.3f}  {r['source']}")

## 4. Load the SLM and compare constrained vs baseline

In [ ]:
from src.generator import Generator
from src.evaluation import grounding_score

gen = Generator()
ctx = retriever.search(q, top_k=TOP_K, method='hybrid')

constrained = gen.generate_constrained(q, ctx)
baseline = gen.generate_baseline(q)

print('CONSTRAINED:\n', constrained)
print('  grounding =', round(grounding_score(constrained, ctx), 3))
print('\nBASELINE:\n', baseline)
print('  grounding vs module =', round(grounding_score(baseline, ctx), 3))

## 5. Full experiment
For the complete metric matrix over your whole QA set, run in a terminal:
```bash
python scripts/3_run_experiments.py
```
then load `outputs/results_per_question.csv` here to make plots for the report.

In [ ]:
import pandas as pd
from src.config import OUTPUTS_DIR
csv = OUTPUTS_DIR / 'results_per_question.csv'
if csv.exists():
    df = pd.read_csv(csv)
    display(df[df.stage == 'generation'].groupby('method')[['grounding','hallucination_rate','latency_s']].mean())
else:
    print('Run scripts/3_run_experiments.py first.')